# Thermal fragmentation — theory and numerical scheme

This notebook documents the physical model, its variational structure, the
discretisation and the algorithm implemented in `problems/thermal.py`.

**Contents**

1. Problem setup
2. Eigenstrain decomposition
3. Energy functional
4. AT1 vs AT2 dissipation
5. Quasi-static formulation
6. Dynamic formulation (Newmark-$\beta$)
7. Viscosity: $c_1$, $c_2$, and the new $c_3$ filter
8. Periodic fragmentation: physical mechanism
9. Spatial discretisation
10. Algorithm pseudo-code
11. Quantities of interest

## 1.  Problem setup

Homogeneous body $\Omega$ in dimension $d\in\{1,2\}$ on an elastic foundation of stiffness $\Lambda^{2}$.  Here the loading is **thermal**: a spatially uniform temperature increase $\theta(t)$ is imposed inside $\Omega$.  The mechanical boundaries are *free*; the only thing that prevents rigid-body motion is the foundation.

- **1D**:  $\Omega = (0, L_x)$.
- **2D**:  $\Omega = (0,L_x)\times(0,L_y)$ meshed with triangles.

Damage is *free* on every boundary — cracks can (and will) nucleate at the edges.

The code now uses two thermal load histories.  The quasi-static branch is linear,

$$
\theta_{\mathrm{qs}}(t)=\theta_{\max}t,
$$

so $\theta=t$ exactly when $\theta_{\max}=1$.  The dynamic branch uses

$$
\tau=\eta t,\qquad
\theta_{\mathrm{dyn}}(t)=\theta_{\max}\frac{\tau}{2}\left[1+\tanh\left(\frac{\tau}{T_0}\right)\right].
$$

For $\tau\gg T_0$ this is nearly linear; near $\tau=0$, the hyperbolic tangent smooths the start of the dynamic load. Larger $\eta$ drives the same computational interval faster through the thermal loading history; smaller $\eta$ slows it down.

**Time-scaling requirement:** To reach the exact same maximum thermal load $\theta_{\max}$ as the quasi-static history, the dynamic simulation time must be inversely scaled by the loading rate:
$$
t_{\mathrm{final}}^{\mathrm{dyn}} = \frac{t_{\mathrm{final}}^{\mathrm{qs}}}{\eta}
$$
Without this adjustment, a small $\eta$ will cause the dynamic simulation to terminate before the target temperature is reached.

## 2.  Eigenstrain decomposition

The total strain $\varepsilon = \tfrac12(\nabla\boldsymbol{u}+\nabla\boldsymbol{u}^\top)$ is split into an elastic part and a *thermal eigenstrain*:

$$
\varepsilon \;=\; \varepsilon^{e} + \theta\,\mathbf{I},
\qquad
\varepsilon^{e}\;=\;\varepsilon - \theta\,\mathbf{I}.
$$

Only the *elastic* part stores energy.  In 1D this collapses to $\varepsilon^{e} = \partial_x u - \theta$.

*Physical interpretation.* When $\theta>0$ the body wants to expand by $\theta L_x$ in 1D (or $\theta\,\mathbf{I}$ everywhere in 2D), but the foundation pulls it back to the reference configuration.  The result is a self-equilibrated stress state, which — once it exceeds the toughness — releases by creating cracks.

## 3.  Energy functional

$$
\mathcal{E}(\boldsymbol{u},\alpha;\theta) =
\underbrace{\int_\Omega \tfrac12 g(\alpha)\,\varepsilon^{e}:\mathbb{C}:\varepsilon^{e}\,d\Omega}_{\mathcal{P}_{\!el}}
+ \underbrace{\int_\Omega \tfrac12\Lambda^{2}|\boldsymbol{u}|^{2}\,d\Omega}_{\mathcal{P}_{\!f}}
+ \underbrace{\int_\Omega \psi_d(\alpha,\nabla\alpha)\,d\Omega}_{\mathcal{S}}.
$$

Kinetic energy (only used in the dynamic run):

$$
\mathcal{K}(\dot{\boldsymbol{u}}) = \int_\Omega \tfrac12 |\dot{\boldsymbol{u}}|^{2}\,d\Omega.
$$

The only difference with the mechanical problem is the substitution $\varepsilon \to \varepsilon^{e} = \varepsilon - \theta\,\mathbf{I}$ in the elastic energy density.  The parameter $\eta$ is now a loading-time scale through $\tau=\eta t$; it is not a viscosity and it no longer multiplies the mass term.  Dissipation is controlled separately by $c_1$, $c_2$, and $c_3$.

Be careful with notation: the repository parameter $\Lambda$ is the foundation stiffness, while $\lambda_{\mathrm{Lame}}$ is the elastic Lame parameter computed from $E_{ref}$ and $\nu$.  The repository default is $\nu=0.3$, so $\lambda_{\mathrm{Lame}}\neq 0$; set $\nu=0$ to recover $\lambda_{\mathrm{Lame}}=0$.  If the thermal problem is run with $\Lambda=0$ and no displacement constraints, the body can expand freely and the equilibrium has a rigid-body/null mode.

## 4.  Fracture-energy density: AT1 vs AT2

Same as in the mechanical theory notebook:

$$
\hat\psi_d(\alpha,\nabla\alpha) = \frac{1}{c_w}\bigl(w(\alpha)+\hat\ell^{\,2}|\nabla\alpha|^{2}\bigr),
$$

with $w(\alpha)=\alpha$, $c_w=8/3$ (AT1) or $w(\alpha)=\alpha^{2}$, $c_w=2$ (AT2).  See `tools/solvers.py`.

The choice is decisive for *when* cracks appear:

- **AT1** has an *elastic phase*: as long as the elastic-strain magnitude stays below a critical threshold, $\alpha=0$ remains a minimiser.  Cracks appear *suddenly* once the threshold is crossed.
- **AT2** has *no elastic phase*: $\alpha$ starts to grow as soon as any strain is applied.  This makes AT2 slightly easier to converge but gives a less sharp force–displacement curve.

## 5.  Quasi-static formulation

At each loading step $t^n$ we minimise $\mathcal{E}(\boldsymbol{u},\alpha;\theta(t^n))$ over $(\boldsymbol{u},\alpha)$ with $\alpha\ge\alpha^{n-1}$ and $\alpha\in[0,1]$.  Alternate minimisation:

$$
\begin{aligned}
\boldsymbol{u}^{n,k+1} &= \arg\min_{\boldsymbol{u}}\,\mathcal{E}(\boldsymbol{u},\alpha^{n,k};\theta^n)
  &&\text{linear elliptic}\\
\alpha^{n,k+1} &= \arg\min_{\alpha^{n-1}\le\alpha\le 1}\,\mathcal{E}(\boldsymbol{u}^{n,k+1},\alpha;\theta^n)
  &&\text{bound-constrained QP}.
\end{aligned}
$$

Same alternate-minimisation kernel as in `problems/dynamic.py` (we re-use `tools.helpers.alt_min_loop`).  Note that *no* Dirichlet BC is applied: equilibrium is well-posed because of the foundation term $\Lambda^{2}|\boldsymbol{u}|^{2}$.

**Iso-stress regime (analytical sanity check).** If we *forbid* damage ($\alpha\equiv0$) and look for a uniform solution $\boldsymbol{u}\equiv 0$ in 1D, the stress is

$$
\bar\sigma(\theta) = -\theta
$$

(compressive when the body is heated and prevented from expanding).  The fragmentation regime appears when this stress exceeds the AT1/AT2 critical value, which depends on $\Lambda$ and $\hat\ell$.

## 6.  Dynamic formulation

Euler–Lagrange equations of $\mathcal{K}-\mathcal{E}$:

$$
\ddot{\boldsymbol{u}} - \nabla\!\cdot\!\bigl(g(\alpha)\,\mathbb{C}:\varepsilon^{e}\bigr) + \Lambda^{2}\,\boldsymbol{u} = \boldsymbol{0},
$$

with the damage variational inequality of section 5.

Time integration: implicit Newmark-$\beta$ with $\beta=1/4$, $\gamma=1/2$ (unconditionally stable for linear problems).  At every step we alternate-minimise on $(\boldsymbol{a}^{n+1},\alpha^{n+1})$:

$$
\begin{aligned}
\boldsymbol{a}^{n+1} &\leftarrow \text{linear elastic equilibrium with }\boldsymbol{u}^{n+1}=\boldsymbol{u}^{n}+\Delta t\,\boldsymbol{v}^{n}+\tfrac12\Delta t^{2}\bigl[(1-2\beta)\boldsymbol{a}^{n}+2\beta\,\boldsymbol{a}^{n+1}\bigr]\\
\alpha^{n+1} &\leftarrow \text{bound-constrained QP at fixed }\boldsymbol{u}^{n+1},\;\alpha\ge\alpha^{n}
\end{aligned}
$$

and finally update $(\boldsymbol{u},\boldsymbol{v},\boldsymbol{a})$ with the standard Newmark formulas.

## 7.  Viscosity: $c_1$, $c_2$, and the new $c_3$ filter

The dynamic thermal solve uses the same Rayleigh-like dissipation potential as the mechanical solve:

$$
\mathcal{Q}(\dot{\boldsymbol{u}},\dot{\alpha})=\int_\Omega\left(\tfrac12 c_1|\dot{\boldsymbol{u}}|^2+c_2\,\psi_0\!\left(\boldsymbol{\varepsilon}(\dot{\boldsymbol{u}})\right)+\tfrac12 c_3\,|\dot{\alpha}|^2\right)d\Omega,\qquad \psi_0(\boldsymbol{\varepsilon})=\tfrac12\lambda_{\mathrm{Lame}}\operatorname{tr}(\boldsymbol{\varepsilon})^2+\mu\,\boldsymbol{\varepsilon}:\boldsymbol{\varepsilon}.
$$

The resulting weak-form contributions are:

- displacement sub-problem:
$$
\int_\Omega c_1\dot{\boldsymbol{u}}\cdot\hat{\boldsymbol{u}}+c_2\left(\mathbb{C}:\boldsymbol{\varepsilon}(\dot{\boldsymbol{u}})\right):\boldsymbol{\varepsilon}(\hat{\boldsymbol{u}})\,d\Omega.
$$

- damage sub-problem:
$$
\int_\Omega c_3\,\dot{\alpha}\,\hat{\alpha}\,d\Omega.
$$

$c_1$ damps local velocity; $c_2$ is a **Kelvin-Voigt** viscous stress proportional to the elastic stiffness, $\boldsymbol{\sigma}_{\mathrm{vis}}=c_2\,\mathbb{C}:\boldsymbol{\varepsilon}(\dot{\boldsymbol{u}})$ (so at the default $\nu=0.3$ it carries a volumetric viscous part); and $c_3$ damps the **damage rate** $\dot{\alpha}$.  The $c_3$ term provides viscous regularisation of the damage evolution: it prevents spuriously fast damage growth during dynamic crack propagation and helps stabilise the alternate-minimisation iterations in the thermal fragmentation regime.  The reported mean stress includes the elastic stress and the $c_2$ Kelvin-Voigt strain-rate stress; $c_3$ is not reported as a Cauchy stress because it acts only on the damage field.

## 8.  Why does the thermal test produce *periodic* fragmentation?

Consider 1D for clarity.  At a given temperature $\theta$ the *uniform* solution $u\equiv 0$ stores an elastic-strain energy density of $\tfrac12\theta^{2}$ everywhere; creating *one* crack relieves the strain in a neighbourhood of size $\sim\hat\ell$ but the foundation immediately rebuilds the stress field away from the crack.  This is the **stress shielding length**:

$$
\ell_{s} \sim \frac{1}{\Lambda},
$$

outside which a *new* crack is energetically favourable.  As $\theta$ continues to ramp, cracks nucleate in successive *generations*, each one spaced roughly $\ell_{s}$ apart.  The competition between $\hat\ell$ (which sets crack width) and $\ell_{s}$ (which sets the spacing) controls the fragmentation pattern.

In 2D the same mechanism gives a quasi-periodic array of crack lines orthogonal to the principal direction of the thermal stress (here, all directions, since the load is isotropic — the actual pattern depends on the boundary).

## 9.  Spatial discretisation

Same conventions as in the mechanical theory notebook: continuous P1 Lagrange elements for $\boldsymbol{u}$ (scalar in 1D, vector in 2D) and $\alpha$ (always scalar).  The 2D mesh is triangular with the `crossed` diagonal layout, removing the spurious anisotropy of the standard `right`/`left` diagonals — crucial for resolving the *crack spacing* in fragmentation.

Mesh refinement rule of thumb: $h \lesssim \hat\ell/2$ (at least four cells across each crack band).

## 10.  Algorithm pseudo-code

```
build mesh, P1 spaces                 # tools.meshing  (no Dirichlet BC)
assemble elastic, fracture, kinetic, and c1/c2/c3 viscous forms

# QUASI-STATIC LOOP
alpha_lb = 0
for ti in (t1,...,tN_qs):
    theta = Theta_qs(ti)
    repeat
        solve linear equilibrium
        solve damage VI (alpha in [alpha_lb, 1])
    until || alpha - alpha_prev || < tol
    alpha_lb := alpha
    record  theta, mean_stress, energies
    if ti in snapshots: record alpha profile

# DYNAMIC LOOP
# Scale final time to match quasi-static thermal load peak
t_final_dyn = t_final_qs / eta
N_dyn = int(t_final_dyn / dt)

u, v, a = 0;  alpha_lb = 0
for step in 1..N_dyn:
    t += dt
    theta = Theta_dyn(t)
    repeat
        solve linear problem for a_new (Newmark)
        solve damage VI
    until convergence
    u_new = u + dt v + 0.5 dt^2 [(1-2 beta) a + 2 beta a_new]
    v_new = v + dt [(1-gamma) a + gamma a_new]
    advance u, v, a;  alpha_lb := alpha
    record theta, mean_stress, energies
    if step in snapshots: append (alpha, u) to Paraview series
```

## 11.  Quantities of interest

For each run we record, at every time step,

- $\theta(t)$ and the *mean stress* $\bar\sigma(t) = \dfrac{1}{|\Omega|}\int_\Omega \sigma\,d\Omega$,
- the elastic, foundation, fracture and kinetic energies,
- the stored dynamic total $\mathcal{K}+\mathcal{P}_{\!el}+\mathcal{P}_{\!f}+\mathcal{S}$ and the cumulative dissipated energy $\mathcal{D}$ as separate curves,
- intermediate damage *snapshots* at a user-specified number of pseudo-times, used by the plotter to show *fragmentation generations*.

`tools/plotting.plot_thermal_run` produces a four-panel figure:

1. mean stress vs $\theta$ (QS vs dyn),
2. QS damage profile snapshots (1D) or final QS damage field (2D),
3. Dynamic damage profile snapshots (1D) or final Dyn damage field (2D),
4. all energy histories overlaid.

The Paraview XDMF series allows interactive inspection of the time evolution of $\alpha$ and $\boldsymbol{u}$ in 2D.

## How to run

```bash
# Single run, 1D, AT2:
python problems/thermal.py

# 2D variant: open problems/thermal.py and set
#   cfg['mesh_parameters']['physics']  = '2D'
#   cfg['solver_parameters']['model']  = 'AT1'
# then re-run.

# Parameter sweep:
python sweep.py thermal
```